# PIQP vs SciPy on a constrained QP

This notebook solves a small convex quadratic program with equality and inequality constraints using `solvers.piqp`, then compares the result with SciPy SLSQP.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "CMakeLists.txt").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "build"))

import numpy as np
from scipy.optimize import minimize
from solvers import piqp

In [2]:
P = np.array([
    [4.0, 1.0, 0.0],
    [1.0, 3.0, 0.5],
    [0.0, 0.5, 2.0],
])
q = np.array([-5.0, -6.0, -3.0])

# Equality: x0 + x1 + x2 = 1.5
Aeq = np.array([[1.0, 1.0, 1.0]])
beq = np.array([1.5])

# PIQP uses G x <= h. This encodes x >= 0.
G = -np.eye(3)
h = np.zeros(3)

def objective(x):
    return 0.5 * x @ P @ x + q @ x

def gradient(x):
    return P @ x + q

settings = piqp.PIQPSettings()
settings.eps_abs = 1e-8
settings.eps_rel = 1e-8
piqp_result = piqp.solve(P, q, Aeq, beq, G, h, settings)
x_piqp = piqp_result["x"]

constraints = [{"type": "eq", "fun": lambda x: Aeq @ x - beq, "jac": lambda x: Aeq}]
bounds = [(0.0, None)] * 3
scipy_result = minimize(
    objective,
    np.full(3, 0.5),
    jac=gradient,
    bounds=bounds,
    constraints=constraints,
    method="SLSQP",
    options={"ftol": 1e-12, "maxiter": 200},
)

print("PIQP status:", piqp_result["status"])
print("PIQP x:", x_piqp)
print("SciPy x:", scipy_result.x)
print("Max error:", np.max(np.abs(x_piqp - scipy_result.x)))
print("Equality residual:", Aeq @ x_piqp - beq)
print("Minimum x:", x_piqp.min())
print("Objective values:", objective(x_piqp), scipy_result.fun)

PIQP status: solved
PIQP x: [0.38732394 1.07042252 0.04225354]
SciPy x: [0.38732395 1.07042252 0.04225353]
Max error: 1.2617661804270597e-08
Equality residual: [-7.69520003e-11]
Minimum x: 0.0422535424471679
Objective values: -6.028169013901338 -6.028169014084506


In [3]:
assert piqp_result["status"] == "solved"
assert scipy_result.success
assert np.allclose(Aeq @ x_piqp, beq, atol=1e-5)
assert x_piqp.min() >= -1e-5
assert np.allclose(x_piqp, scipy_result.x, rtol=1e-4, atol=1e-4)